# In-Place TTT - Setup & Train Qwen3-4B trên RunAI (H200)

Notebook này hướng dẫn từng bước setup và train In-Place TTT với Qwen3-4B.

**Yêu cầu:** GPU H200, model Qwen3-4B đã upload sẵn trên RunAI.

## Cell 0: Kiểm tra GPU

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Cell 1: Cài đặt dependencies

Cài VeOmni framework + các thư viện cần thiết. Chạy cell này **một lần** khi mới setup.

In [ ]:
# Cài PyTorch + CUDA (nếu chưa có)
!pip install torch==2.8.0+cu128 torchvision==0.23.0+cu128 torchaudio==2.8.0+cu128 \
    --index-url https://download.pytorch.org/whl/cu128

# Cài flash-attention (tăng tốc + tiết kiệm VRAM)
!pip install flash-attn --no-build-isolation

# Cài các dependencies chính
!pip install transformers==4.57.3 datasets einops wandb opt-einsum blobfile tiktoken psutil timm

In [ ]:
# Cài VeOmni framework (từ source)
!pip install git+https://github.com/ByteDance-Seed/VeOmni.git

## Cell 2: Clone repo In-Place-TTT

Nếu đã upload repo lên RunAI thì bỏ qua cell này.

In [ ]:
import os

WORK_DIR = os.path.expanduser("~/In-Place-TTT")

if not os.path.exists(WORK_DIR):
    !git clone https://github.com/ByteDance-Seed/In-Place-TTT.git {WORK_DIR}
    print(f"Cloned to {WORK_DIR}")
else:
    print(f"Repo already exists at {WORK_DIR}")

os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")

## Cell 3: Cấu hình đường dẫn

**QUAN TRỌNG:** Sửa các đường dẫn bên dưới cho đúng với vị trí model và data trên RunAI.

In [ ]:
# ============================================================
# SỬA CÁC ĐƯỜNG DẪN NÀY CHO ĐÚNG VỚI MÔI TRƯỜNG RUNAI
# ============================================================

# Đường dẫn tới model Qwen3-4B đã upload
MODEL_PATH = "/path/to/your/Qwen3-4B"

# Đường dẫn tới training data (plaintext format)
DATA_PATH = "/path/to/your/training_data"

# Đường dẫn output (nơi lưu checkpoint)
OUTPUT_DIR = "/path/to/your/output"

# ============================================================

# Verify paths
for name, path in [("MODEL_PATH", MODEL_PATH), ("DATA_PATH", DATA_PATH)]:
    if os.path.exists(path):
        print(f"{name}: {path} [OK]")
    else:
        print(f"{name}: {path} [NOT FOUND - hãy sửa lại!]")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"OUTPUT_DIR: {OUTPUT_DIR} [OK]")

## Cell 4: Tạo config training cho Qwen3-4B

Qwen3-4B có **36 layers**, `hidden_size=2560`. Config TTT layers phân bố đều trong 36 layers.

In [ ]:
import yaml

config = {
    "model": {
        "model_path": MODEL_PATH,
        "foundation": {
            # Qwen3-4B có 36 layers → chọn 6 layers phân bố đều
            "ttt_layers": [0, 6, 12, 18, 24, 30],
            "ttt_mode": True,
            "ttt_proj": True,
            "ttt_lr": 3,
            "ttt_chunk": 4096,
        },
    },
    "data": {
        "train_path": DATA_PATH,
        "train_size": 20000000000,
        "dataloader_type": "native",
        "datasets_type": "iterable",
        "data_type": "plaintext",
        "max_seq_len": 65536,       # H200 đủ VRAM cho 64K context
        "text_keys": "content_split",
        "drop_last": True,
    },
    "train": {
        "output_dir": OUTPUT_DIR,
        "data_parallel_mode": "fsdp2",
        "ulysses_parallel_size": 1,
        "global_batch_size": 1,      # Single GPU
        "micro_batch_size": 1,
        "rmpad": False,
        "use_wandb": False,          # Tắt wandb (bật lại nếu muốn logging)
        "max_steps": 5000,
        "rmpad_with_pos_ids": False,
        "dyn_bsz_margin": 0,
        "dyn_bsz": False,
        "optimizer": "adamw",
        "lr": 5.0e-6,
        "lr_warmup_ratio": 0.02,
        "lr_decay_style": "cosine",
        "lr_decay_ratio": 0.90,
        "weight_decay": 0.1,
        "max_grad_norm": 1.0,
        "enable_mixed_precision": True,
        "enable_gradient_checkpointing": True,
        "enable_full_shard": True,
        "init_device": "meta",
        "ckpt_manager": "dcp",
        "save_steps": 500,
        "save_hf_weights": True,
    },
}

config_path = os.path.join(WORK_DIR, "configs/pretrain/qwen3_4b_longct.yaml")
os.makedirs(os.path.dirname(config_path), exist_ok=True)

with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f"Config saved to: {config_path}")
print("---")
with open(config_path) as f:
    print(f.read())

## Cell 5: Verify model files

Kiểm tra model Qwen3-4B đã upload đúng chưa.

In [ ]:
import json

# Kiểm tra model files
model_files = os.listdir(MODEL_PATH)
print("Model files:")
for f in sorted(model_files):
    size = os.path.getsize(os.path.join(MODEL_PATH, f)) / 1e9
    print(f"  {f} ({size:.2f} GB)" if size > 0.01 else f"  {f}")

# Verify config
config_file = os.path.join(MODEL_PATH, "config.json")
if os.path.exists(config_file):
    with open(config_file) as f:
        model_config = json.load(f)
    print(f"\nModel config:")
    print(f"  num_hidden_layers: {model_config.get('num_hidden_layers')}")
    print(f"  hidden_size: {model_config.get('hidden_size')}")
    print(f"  intermediate_size: {model_config.get('intermediate_size')}")
    print(f"  model_type: {model_config.get('model_type')}")
    
    # Verify TTT layers phù hợp
    n_layers = model_config.get('num_hidden_layers', 0)
    ttt_layers = [0, 6, 12, 18, 24, 30]
    if all(l < n_layers for l in ttt_layers):
        print(f"\n  ttt_layers {ttt_layers} OK (model có {n_layers} layers)")
    else:
        print(f"\n  WARNING: ttt_layers có index >= {n_layers}! Cần điều chỉnh.")
else:
    print("\nWARNING: config.json not found!")

## Cell 6: Chạy Training

Dùng `torchrun` để chạy training trên single GPU.

**Lưu ý:** Cell này chạy lâu (hàng giờ tùy data). Xem log trực tiếp bên dưới.

In [ ]:
os.chdir(WORK_DIR)

# Set environment
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TORCH_NCCL_AVOID_RECORD_STREAMS"] = "1"
os.environ["PYTHONPATH"] = f"{WORK_DIR}:{os.environ.get('PYTHONPATH', '')}"

config_file = "configs/pretrain/qwen3_4b_longct.yaml"

!torchrun --standalone --nproc-per-node=1 tasks/train_torch.py --config {config_file}

## Cell 7: Convert checkpoint sang HuggingFace format

Sau khi train xong, convert DCP checkpoint sang HF format để dùng cho inference.

In [ ]:
# Tìm checkpoint mới nhất
import glob

ckpt_dirs = sorted(glob.glob(os.path.join(OUTPUT_DIR, "checkpoints/global_step_*")))
if ckpt_dirs:
    latest_ckpt = ckpt_dirs[-1]
    print(f"Latest checkpoint: {latest_ckpt}")
    
    # Check xem đã có hf_ckpt chưa
    hf_ckpt = os.path.join(latest_ckpt, "hf_ckpt")
    if os.path.exists(hf_ckpt):
        print(f"HF checkpoint đã có sẵn: {hf_ckpt}")
    else:
        print("Cần convert...")
        !python scripts/merge_dcp_to_hf.py \
            --dcp_path {latest_ckpt} \
            --output_path {hf_ckpt} \
            --model_path {MODEL_PATH}
        print(f"Converted to: {hf_ckpt}")
else:
    print("Không tìm thấy checkpoint nào. Hãy chạy training trước (Cell 6).")

## Cell 8: Test Inference

Load model đã train và thử generate.

In [ ]:
import sys
sys.path.insert(0, WORK_DIR)

import torch
from transformers import AutoTokenizer

# Import custom TTT model
import inference_model  # noqa: F401
from inference_model.hf_qwen3.modeling_qwen3 import Qwen3ForCausalLM
from inference_model.hf_qwen3.configuration_qwen3 import Qwen3Config

# Đường dẫn tới HF checkpoint đã convert
# Sửa lại cho đúng nếu khác
HF_CKPT_PATH = hf_ckpt if 'hf_ckpt' in dir() else os.path.join(OUTPUT_DIR, "checkpoints/global_step_5000/hf_ckpt")

print(f"Loading model from: {HF_CKPT_PATH}")
config = Qwen3Config.from_pretrained(HF_CKPT_PATH)
config.ttt_mode = True  # Bật TTT cho inference

model = Qwen3ForCausalLM.from_pretrained(
    HF_CKPT_PATH,
    config=config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

print("Model loaded!")
print(f"TTT layers: {config.ttt_layers}")
print(f"TTT mode: {config.ttt_mode}")

In [ ]:
# Test generation
prompt = "Explain the concept of test-time training in one paragraph."

messages = [{"role": "user", "content": prompt}]
input_ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
input_ids = input_ids.to(model.device)

with torch.no_grad():
    output = model.generate(
        input_ids=input_ids,
        attention_mask=torch.ones_like(input_ids),
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

response = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True)
print(f"Prompt: {prompt}")
print(f"\nResponse: {response}")

## Troubleshooting

| Lỗi | Giải pháp |
|------|----------|
| `ModuleNotFoundError: No module named 'veomni'` | Chạy lại Cell 1, cài VeOmni |
| `ModuleNotFoundError: No module named 'hf_models'` | Đảm bảo `os.chdir(WORK_DIR)` và `PYTHONPATH` đúng |
| `CUDA out of memory` | Giảm `max_seq_len` (thử 32768 hoặc 16384) |
| `ImportError: flash_attn` | Chạy `pip install flash-attn --no-build-isolation` |
| `ttt_layers index out of range` | Đảm bảo tất cả index trong `ttt_layers` < `num_hidden_layers` (36 cho Qwen3-4B) |
| Checkpoint not found | Kiểm tra `OUTPUT_DIR` và `save_steps` trong config |